# P3 — Futebol Analytics com YOLO (pipeline completo no Colab)
Ambiente: **GPU T4** (Ambiente de execução → Alterar tipo → T4). Rode as células em ordem.

## 0. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

REPO = "https://github.com/SEU_USUARIO/p3-futebol-yolo.git"   # <- troque
!git clone -q $REPO /content/p3 || (cd /content/p3 && git pull -q)
%cd /content/p3/src
!pip install -q -r requirements.txt

In [ ]:
# Parâmetros do trecho (registre os mesmos no README)
URL      = "https://www.youtube.com/watch?v=VIDEO_ID"   # <- troque
INICIO   = 12.0   # minutos
FIM      = 16.0   # minutos (3–5 min de trecho)
JOGO     = "jogo1"
DRIVE    = "/content/drive/MyDrive/P3_futebol"          # vídeos ficam aqui, fora do git
SAIDA    = f"../saidas/{JOGO}"
MODELO   = "yolov8m.pt"
!mkdir -p "$DRIVE" $SAIDA

## 1. Aquisição
Se o YouTube bloquear o Colab ("confirm you're not a bot"), baixe no PC com
`python baixar_video.py URL --inicio 12 --fim 16 --saida videos`, suba o `.mp4` para `DRIVE` e pule para a célula seguinte.

In [ ]:
!python baixar_video.py "$URL" --inicio $INICIO --fim $FIM --saida "$DRIVE"
!ls -lh "$DRIVE"

In [ ]:
import glob, cv2
VIDEO = sorted(glob.glob(f"{DRIVE}/*.mp4"))[-1]
cap = cv2.VideoCapture(VIDEO)
print(VIDEO, int(cap.get(3)), "x", int(cap.get(4)), "@", cap.get(5), "fps |", int(cap.get(7)), "quadros")   # Q1.2

## 2. Comparação de modelos (Q2.1) — mesmo subtrecho de 30 s

In [ ]:
for m in ["yolov8n.pt", "yolov8s.pt", "yolov8m.pt"]:
    tag = m.split('.')[0]
    !python detectar_rastrear.py --video "$VIDEO" --saida ../saidas/cmp_{tag} --modelo {m} --inicio 0 --fim 0.5 --sem-video
!python comparar_modelos.py ../saidas/cmp_yolov8n ../saidas/cmp_yolov8s ../saidas/cmp_yolov8m --csv ../saidas/comparacao_modelos.csv

In [ ]:
# Q2.3 — efeito de imgsz e conf-bola (yolov8m, 30 s)
for imgsz in [640, 1280]:
    for cb in [0.05, 0.10, 0.25]:
        !python detectar_rastrear.py --video "$VIDEO" --saida ../saidas/cmp_img{imgsz}_cb{cb} --modelo $MODELO --imgsz {imgsz} --conf-bola {cb} --inicio 0 --fim 0.5 --sem-video
!python comparar_modelos.py ../saidas/cmp_img*_cb* --csv ../saidas/comparacao_imgsz_conf.csv

## 3. Trecho completo: detecção + ByteTrack + times

In [ ]:
!python detectar_rastrear.py --video "$VIDEO" --saida $SAIDA --modelo $MODELO --imgsz 1280
!cat $SAIDA/meta.json

In [ ]:
# Q3.3 — teste com k=3 (separa árbitro/goleiros?) em 1 min
!python detectar_rastrear.py --video "$VIDEO" --saida ../saidas/{JOGO}_k3 --modelo $MODELO --k 3 --inicio 0 --fim 1

## 4. Bola (Kalman) e homografia

In [ ]:
!python bola_kalman.py --saida $SAIDA

**Calibração sem janela:** a célula abaixo mostra o `frame0.jpg` com grade de pixels.
Anote (u, v) de ≥ 4 marcos bem espalhados (nomes em `campo.MARCOS`) e preencha a célula seguinte.

In [ ]:
import matplotlib.pyplot as plt, campo
img = cv2.cvtColor(cv2.imread(f"{SAIDA}/frame0.jpg"), cv2.COLOR_BGR2RGB)
plt.figure(figsize=(18, 10)); plt.imshow(img)
plt.xticks(range(0, img.shape[1], 50), rotation=90); plt.yticks(range(0, img.shape[0], 50)); plt.grid(alpha=.35)
plt.show()
print(list(campo.MARCOS))

In [ ]:
import json
calib = {
    "pixels": [[640, 112], [598, 361], [520, 702], [1180, 230]],          # <- troque
    "marcos": ["meio_sup", "centro", "meio_inf", "ga_dir_sup"],           # <- troque
}
json.dump(calib, open(f"{SAIDA}/calibracao.json", "w"))
!python homografia.py calibrar --saida $SAIDA --sem-janela
!python homografia.py camera   --saida $SAIDA
!python homografia.py projetar --saida $SAIDA

In [ ]:
from IPython.display import Image
Image(f"{SAIDA}/calibracao_grade.jpg") if __import__('os').path.exists(f"{SAIDA}/calibracao_grade.jpg") else print("ver saída do calibrar")

## 5. Métricas, sensibilidade e métricas táticas

In [ ]:
!python metricas.py --saida $SAIDA          # confira a direção de ataque; se errada: --ataque-time0 direita|esquerda
!python sensibilidade_raio.py --saida $SAIDA --raios 1 1.5 2 2.5 3
!python metricas_taticas.py --saida $SAIDA

### Validação manual (Q5.3) — obrigatória (−15% sem ela)
Assista ao `rastreado.mp4` e preencha `saidas/<jogo>/passes_anotados.csv` (`t_s,time,sucesso`).

In [ ]:
!python validar_passes.py --saida $SAIDA --anotado $SAIDA/passes_anotados.csv

## 6. eXP

In [ ]:
!python xpass.py --saida $SAIDA --what-if

## 7. Vídeo da apresentação + cópia de resultados

In [ ]:
!python video_minimapa.py --saida $SAIDA --inicio-s 30 --dur-s 90
!cp $SAIDA/rastreado.mp4 $SAIDA/demo_minimapa.mp4 "$DRIVE"/
!mkdir -p ../figuras/$JOGO && cp $SAIDA/figuras/*.png ../figuras/$JOGO/
!ls ../figuras/$JOGO

In [ ]:
# Baixa saídas (sem vídeos) para commitar no repositório
!cd .. && zip -qr /content/saidas_{JOGO}.zip saidas figuras -x "*.mp4"
from google.colab import files; files.download(f"/content/saidas_{JOGO}.zip")